# Accessing the MOJ/ISA data via API calls

Created: 2026-09-28

The workbooks of immediate interest are in the [statistics on registered foreign residents that are provided by the Immigration Services Agency](https://www.moj.go.jp/isa/policies/statistics/toukei_ichiran_touroku.html). These semiannual datasets each contain two tables that can be accessed via the API. In order to develop functionality that produces dataframes that are consistent, this notebook downloads the API tables using estatjp's API functions. The two tables have the following urls:

In [2]:
# Foreign residents by nationality/region and visa type (table 14-12-01-1):
visaurl = 'http://api.e-stat.go.jp/rest/3.0/app/getSimpleStatsData?appId=&lang=J&statsDataId=0004019020&metaGetFlg=Y&cntGetFlg=N&explanationGetFlg=Y&annotationGetFlg=Y&sectionHeaderFlg=1&replaceSpChars=0'
# Foreign residents by nationality/region, age and gender(table 14-12-12-1):
agegenderurl = 'http://api.e-stat.go.jp/rest/3.0/app/getSimpleStatsData?appId=&lang=J&statsDataId=0004023760&metaGetFlg=Y&cntGetFlg=N&explanationGetFlg=Y&annotationGetFlg=Y&sectionHeaderFlg=1&replaceSpChars=0'


## Import packages

In [2]:
import os
from dotenv import load_dotenv
import urllib.parse as p
from estatjp import exceptions as xs
from estatjp import api
import pandas as pd
import datetime

## Look at foreign residents by nationality/region and visa type (table 14-12-01-1)

In [11]:
try:        
    dfsvisa = api.get_csv_data(visaurl,description=datetime.datetime.now())
    print(dfsvisa.get('Description'))
    print(dfsvisa.get('Header'))
    print(dfsvisa.get('Main'))
except Exception as e:
    print(type(e))
    print(e.args)
    print(e.with_traceback)

2026-09-28 11:41:31.830825
                       0                              1                   2
0                 RESULT                            NaN                 NaN
1                 STATUS                              0                 NaN
2              ERROR_MSG                     正常に終了しました。                 NaN
3                   DATE  2026-09-28T11:41:41.092+09:00                 NaN
4             RESULT_INF                            NaN                 NaN
5           TOTAL_NUMBER                         191475                 NaN
6            FROM_NUMBER                              1                 NaN
7              TO_NUMBER                         100000                 NaN
8               NEXT_KEY                         100001                 NaN
9              TABLE_INF                     0004019020                 NaN
10             STAT_NAME                       00250012   在留外国人統計（旧登録外国人統計）
11               GOV_ORG                          00250      

## Look at foreign residents by nationality/region, age and gender(table 14-12-12-1)

In [21]:
try:        
    dfsagegender = api.get_csv_data(agegenderurl,description=datetime.datetime.now())
    print(dfsagegender.get('Description'))
    print(dfsagegender.get('Header'))
    print(dfsagegender.get('Main'))
except Exception as e:
    print(type(e))
    print(e.args)
    print(e.with_traceback)

2026-09-28 12:13:39.214573
                       0                              1                   2
0                 RESULT                            NaN                 NaN
1                 STATUS                              0                 NaN
2              ERROR_MSG                     正常に終了しました。                 NaN
3                   DATE  2026-09-28T12:13:49.198+09:00                 NaN
4             RESULT_INF                            NaN                 NaN
5           TOTAL_NUMBER                        1533977                 NaN
6            FROM_NUMBER                              1                 NaN
7              TO_NUMBER                         100000                 NaN
8               NEXT_KEY                         100001                 NaN
9              TABLE_INF                     0004023760                 NaN
10             STAT_NAME                       00250012   在留外国人統計（旧登録外国人統計）
11               GOV_ORG                          00250      

## Observations:

1. The API calls access time series covering the entire datasets starting from June 2012.
2. The date of the releases are in a 'time_code' column and a '集計時点（半期毎）' column.
3. The country codes for 'cat01_code' in the ISA statistics CANNOT be matched the 'area_code' in the primary commodity statistics below.

## Extracting some potentially useful data

### cat01_code   在留資格

In [18]:
dfvisa = dfsvisa.get("Main")
type(dfvisa)
a_df=dfvisa.drop_duplicates(['cat01_code'])[['cat01_code','在留資格']]
a_df

,cat01_code,在留資格
0,1010,総数
5477,1040,教授
10954,1050,芸術
16431,1060,宗教
21908,1070,報道
27385,1080,高度専門職合計
28418,1090,高度専門職１号イ
32896,1100,高度専門職１号ロ
37374,1110,高度専門職１号ハ
41852,1120,高度専門職２号


### cat01_code 国籍・地域

In [41]:
dfagegender = dfsagegender.get("Main")
b_df=dfagegender.drop_duplicates(['cat01_code'])[['cat01_code','国籍・地域']]
b_df


,cat01_code,国籍・地域
0,0000,総数
7577,1000,アジア
15154,1010,アフガニスタン
22731,1020,アラブ首長国連邦
30308,1030,イエメン
...,...,...
1496092,6120,パラオ
1503669,6130,フィジー
1511246,6140,マーシャル
1518823,6150,ミクロネシア


### select regions

In [39]:
import re
dfb = dfagegender[dfagegender['cat01_code'].str.contains('000$',regex=True)][['cat01_code','国籍・地域']]
dfb.drop_duplicates()

,cat01_code,国籍・地域
0,0000,総数
7577,1000,アジア
316740,2000,ヨーロッパ
720629,3000,アフリカ
1132396,4000,北アメリカ
1314244,5000,南アメリカ
1412745,6000,オセアニア
1526400,7000,無国籍


## Look at possible English terminology sources

In [ ]:
tradeurl = 'http://api.e-stat.go.jp/rest/3.0/app/getSimpleStatsData?cdCat02=120&cdCat01=00000000%2C30000000%2C50000000&cdTime=2025000000&appId=&lang=E&statsDataId=0003425295&metaGetFlg=Y&cntGetFlg=N&explanationGetFlg=Y&annotationGetFlg=Y&sectionHeaderFlg=1&replaceSpChars=0'
try:        
    dfstrade = api.get_csv_data(tradeurl,description=datetime.datetime.now())
    print(dfstrade.get('Description'))
    print(dfstrade.get('Header'))
    print(dfstrade.get('Main'))
except Exception as e:
    print(type(e))
    print(e.args)
    print(e.with_traceback)

## Discovered resources

### World Bank [World Development Indicators](https://databank.worldbank.org/reports.aspx?source=2&series=SM.POP.TOTL.ZS&country=#)

### Center for Immigration Studies [Immigration Data Portal](https://cis.org/Immigration-Statistics-Data-Portal)

### International Organization for Migration [Migration data](https://www.iom.int/migration-data)

### [Migration Data Portal](https://www.migrationdataportal.org/)